01_ingest_openfda

In [0]:
%python
import requests
import pandas as pd

response = requests.get(
    "https://api.fda.gov/drug/event.json?limit=100"
)

data = response.json()

pdf = pd.json_normalize(data["results"])

display(pdf.head())

Convert to Spark

In [0]:
%python
spark_df = spark.createDataFrame(pdf)

Keep useful fields:

In [0]:
%python
from pyspark.sql.functions import col

openfda_df = spark_df.select(
    col("safetyreportid"),
    col("receivedate"),
    col("serious"),
    col("occurcountry")
)

Save Bronze table:

In [0]:
%sql
USE CATALOG etl_healthcare_project;

CREATE SCHEMA IF NOT EXISTS healthcare;

USE SCHEMA healthcare;

In [0]:
%sql
SHOW SCHEMAS IN etl_healthcare_project;

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
%python
openfda_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(
        "etl_healthcare_project.healthcare.bronze_openfda"
    )

In [0]:
%sql
SHOW SCHEMAS IN etl_healthcare_project;

02_ingest_clinical_trials

In [0]:
%python
import requests

url = "https://clinicaltrials.gov/api/v2/studies?query.term=diabetes&pageSize=100"

response = requests.get(url)

print(response.status_code)
data = response.json()

print(data.keys())

Convert to DataFrame

In [0]:
%python
import pandas as pd

studies = data.get("studies", [])

pdf = pd.json_normalize(studies)

display(pdf.head())

Convert to Spark

In [0]:
%python
spark_df = spark.createDataFrame(pdf)
display(spark_df)


Save Bronze Table

In [0]:
%python
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("etl_healthcare_project.healthcare.bronze_trials")

03_ingest_cdc

In [0]:
%python
import requests
import pandas as pd

url = "https://data.cdc.gov/resource/8xkx-amqh.json?$limit=100"

response = requests.get(url)

print("Status Code:", response.status_code)

data = response.json()

pdf = pd.DataFrame(data)

display(pdf.head())

Inspect the schema

In [0]:
%python
print(pdf.columns)
print(pdf.dtypes)

Convert to Spark

In [0]:
%python
spark_df = spark.createDataFrame(pdf)

display(spark_df)

Create CDC Bronze Table

In [0]:
%python
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("etl_healthcare_project.healthcare.bronze_cdc")

In [0]:
%sql
SELECT *
FROM etl_healthcare_project.healthcare.bronze_cdc
LIMIT 10;

LOAD ALL BRONZE TABLES

In [0]:
%python
from pyspark.sql.functions import *

openfda = spark.table("etl_healthcare_project.healthcare.bronze_openfda")
cdc = spark.table("etl_healthcare_project.healthcare.bronze_cdc")
trial=spark.table("etl_healthcare_project.healthcare.bronze_trials")

06_silver_layer

OPENFDA SILVER

In [0]:
%python
silver_openfda = openfda \
    .withColumnRenamed("serious", "is_serious") \
    .withColumn("is_serious", col("is_serious").cast("int")) \
    .withColumn("country", upper(col("occurcountry"))) \
    .dropDuplicates()

In [0]:
%python
silver_openfda.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("etl_healthcare_project.healthcare.silver_openfda")

CDC SILVER

In [0]:
%python
silver_clinical_trials = spark.read.table(
    "etl_healthcare_project.healthcare.silver_clinical_trials"
)

In [0]:
%python
from pyspark.sql.functions import *

silver_cdc = cdc \
    .withColumn("state", upper(col("recip_state"))) \
    .withColumn("metro_status", col("metro_status")) \
    .dropDuplicates()

In [0]:
%python
silver_cdc.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("etl_healthcare_project.healthcare.silver_cdc")

Silver CDC Clinical Trial

In [0]:
%python
from pyspark.sql.functions import *

silver_clinical_trials = trial \
    .withColumnRenamed("protocolSection.identificationModule.nctId", "trial_id") \
    .withColumnRenamed("protocolSection.identificationModule.briefTitle", "trial_title") \
    .withColumnRenamed("protocolSection.statusModule.overallStatus", "trial_status") \
    .withColumnRenamed("protocolSection.designModule.studyType", "study_type") \
    .withColumnRenamed("protocolSection.designModule.enrollmentInfo.count", "enrollment_count") \
    .withColumn(
        "trial_status",
        upper(col("trial_status"))
    ) \
    .withColumn(
        "study_type",
        upper(col("study_type"))
    ) \
    .withColumn(
        "enrollment_count",
        col("enrollment_count").cast("int")
    ) \
    .dropDuplicates()

In [0]:
%python
silver_clinical_trials.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(
        "etl_healthcare_project.healthcare.silver_clinical_trials"
    )

Validate Your Silver Tables

In [0]:
%python
display(silver_openfda.limit(5))
display(silver_cdc.limit(5))
display(silver_clinical_trials.limit(5))

Build GOLD LAYER (Business Metrics)

CDC Public Health Metrics

In [0]:
%python
cdc_gold = silver_cdc.groupBy("state") \
    .agg(
        avg("fully_vaccinated_pct").alias("vaccination_rate"),
        avg("booster_pct").alias("booster_rate"),
        sum("dose1_count").alias("total_dose1")
    )

FDA Drug Safety Metrics

In [0]:
%python
fda_gold = silver_openfda.groupBy("country") \
    .agg(
        count("*").alias("total_reports"),
        sum(when(col("is_serious") == 1, 1).otherwise(0)).alias("serious_cases")
    ) \
    .withColumn(
        "serious_rate",
        col("serious_cases") / col("total_reports")
    )

Clinical Trials Metrics

In [0]:
%python
trials_gold = silver_clinical_trials.groupBy("study_type") \
    .agg(
        count("*").alias("total_trials"),
        avg("enrollment_count").alias("avg_enrollment")
    )

Unified Healthcare View

In [0]:
%python
healthcare_overview = cdc_gold.alias("cdc") \
    .join(
        fda_gold.alias("fda"),
        col("cdc.state") == col("fda.country"),
        "outer"
    )

Build KPI Tables for Dashboard

In [0]:
%python
cdc_gold = silver_cdc.groupBy("recip_state") \
    .agg(
        avg("series_complete_pop_pct").alias("vaccination_rate"),
        avg("booster_doses_vax_pct").alias("booster_rate"),
        sum("series_complete_yes").alias("total_fully_vaccinated")
    )

Create Visual Data Tables

CDC Trend Table

In [0]:
%python
display(cdc_gold.orderBy(desc("vaccination_rate")))

FDA Severity Table

In [0]:
%python
display(fda_gold.orderBy(desc("serious_rate")))

Clinical Trials Table

In [0]:
%python
display(trials_gold.orderBy(desc("avg_enrollment")))

In [0]:
%python
display(fda_gold.selectExpr("sum(total_reports) as total_fda_reports"))
display(fda_gold.selectExpr("avg(serious_rate) as avg_serious_rate"))
display(cdc_gold.selectExpr("avg(vaccination_rate) as avg_vaccination_rate"))
display(trials_gold.selectExpr("sum(total_trials) as total_trials"))
display(trials_gold.selectExpr("avg(avg_enrollment) as avg_enrollment"))